# Describing the three Netflix data variables

This notebook explores `netflix_titles.csv` for the research question: **How well can a Netflix title's genre and description predict its age rating?**

| CSV column | Meaning | Statistical type | Role in the project |
| --- | --- | --- | --- |
| `listed_in` | One or more genre/category labels assigned to a title | Categorical, nominal, multiple labels per title | Input feature |
| `description` | A short text synopsis of a title | Unstructured text | Input feature |
| `rating` | The recorded age/audience classification | Categorical; different rating systems occur in the same column | Target variable |

Each row represents one Netflix movie or TV show. Summaries below use the original rows and do not remove or alter records.

**To run:** open this notebook from the repository folder, keep `netflix_titles.csv` beside it, use a Python kernel with pandas installed, and choose **Run All**. Saved outputs are included for viewing on GitHub.


In [1]:
from pathlib import Path
import pandas as pd

csv_path = Path("netflix_titles.csv")
df = pd.read_csv(csv_path)
columns = ["listed_in", "description", "rating"]
data = df[columns].copy()

print(f"Dataset: {df.shape[0]:,} titles and {df.shape[1]} columns")
print("\nFirst five titles and their three variables:")
print(df[["title"] + columns].head().to_string(index=False))


Dataset: 8,807 titles and 12 columns

First five titles and their three variables:
                title                                                     listed_in                                                                                                                                              description rating
 Dick Johnson Is Dead                                                 Documentaries As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.  PG-13
        Blood & Water               International TV Shows, TV Dramas, TV Mysteries      After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth.  TV-MA
            Ganglands Crime TV Shows, International TV Shows, TV Action & Adventure       To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of rob

## 1. Overall data quality

The table distinguishes missing values from empty or whitespace-only strings. A stored string type does not mean a variable is numerical. The unique count for `listed_in` counts complete genre combinations; individual genres are counted separately below.


In [2]:
blank_counts = data.apply(
    lambda column: column.fillna("").str.strip().eq("").sum() - column.isna().sum()
)
summary = pd.DataFrame({
    "Stored dtype": data.dtypes.astype(str),
    "Non-missing": data.notna().sum(),
    "Missing": data.isna().sum(),
    "Missing (%)": data.isna().mean().mul(100).round(2),
    "Blank (non-missing)": blank_counts,
    "Unique non-missing": data.nunique(),
})
print(summary.to_string())
print("\nCategorical/text descriptive statistics:")
print(data.describe(include="all").to_string())


            Stored dtype  Non-missing  Missing  Missing (%)  Blank (non-missing)  Unique non-missing
listed_in            str         8807        0         0.00                    0                 514
description          str         8807        0         0.00                    0                8775
rating               str         8803        4         0.05                    0                  17

Categorical/text descriptive statistics:
                           listed_in                                                                                                                                                description rating
count                           8807                                                                                                                                                       8807   8803
unique                           514                                                                                                                        

## 2. Genre: `listed_in`

A title can belong to several genres. For example, `Dramas, International Movies` contains two labels. Splitting on commas counts each individual genre. Percentages use the number of titles with at least one nonblank genre; they can add to more than 100% because titles can have several genres.


In [3]:
genre_lists = data["listed_in"].dropna().str.split(",").apply(
    lambda labels: list(dict.fromkeys(label.strip() for label in labels if label.strip()))
)
genre_lists = genre_lists[genre_lists.str.len().gt(0)]
individual_genres = genre_lists.explode()
genre_counts = individual_genres.value_counts()
genre_table = pd.DataFrame({
    "Title count": genre_counts,
    "Titles with a genre (%)": genre_counts.div(len(genre_lists)).mul(100).round(2),
})

print(f"Distinct genre combinations: {data['listed_in'].nunique()}")
print(f"Distinct individual genres: {individual_genres.nunique()}")
print("\nTen most common individual genres:")
print(genre_table.head(10).to_string())
print("\nNumber of genres per title:")
print(genre_lists.str.len().describe().round(2).to_string())


Distinct genre combinations: 514
Distinct individual genres: 42

Ten most common individual genres:
                          Title count  Titles with a genre (%)
listed_in                                                     
International Movies             2752                    31.25
Dramas                           2427                    27.56
Comedies                         1674                    19.01
International TV Shows           1351                    15.34
Documentaries                     869                     9.87
Action & Adventure                859                     9.75
TV Dramas                         763                     8.66
Independent Movies                756                     8.58
Children & Family Movies          641                     7.28
Romantic Movies                   616                     6.99

Number of genres per title:
count    8807.00
mean        2.19
std         0.78
min         1.00
25%         2.00
50%         2.00
75%         3

## 3. Description: `description`

Descriptions are free text rather than numerical measurements. Word count and character count provide simple numerical summaries of their length. Here, a word is a whitespace-separated token; punctuation remains attached. These length summaries describe the text, but do not measure its meaning or suitability for an age group.


In [4]:
descriptions = data["description"].dropna()
description_lengths = pd.DataFrame({
    "Word count": descriptions.str.split().str.len(),
    "Character count": descriptions.str.len(),
})
print(description_lengths.describe().round(2).to_string())
print(f"\nUnique descriptions: {descriptions.nunique():,}")
print(f"Repeated descriptions beyond first occurrence: {descriptions.duplicated().sum():,}")
print("\nExample descriptions:")
for title, description in df[["title", "description"]].dropna().head(3).itertuples(index=False, name=None):
    print(f"- {title}: {description}")


       Word count  Character count
count     8807.00          8807.00
mean        23.88           143.30
std          2.84            10.34
min         10.00            61.00
25%         22.00           140.00
50%         24.00           146.00
75%         26.00           149.00
max         48.00           248.00

Unique descriptions: 8,775
Repeated descriptions beyond first occurrence: 32

Example descriptions:
- Dick Johnson Is Dead: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.
- Blood & Water: After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth.
- Ganglands: To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war.


## 4. Age rating: `rating`

The target is a category such as `TV-MA`, `TV-14`, or `PG-13`. The dataset combines movie and television rating systems, so the original labels should not be treated as a single numerical scale. `NR` and `UR` are unrated labels rather than age thresholds.

The frequency table includes missing values and uses all dataset rows as its percentage denominator. The checks also identify duration-like values (for example, `74 min`) and labels outside the project's expected set. They are reported for review, without silently deleting them or assigning an age rating.


In [5]:
rating_counts = data["rating"].fillna("<MISSING>").value_counts()
rating_table = pd.DataFrame({
    "Title count": rating_counts,
    "All titles (%)": rating_counts.div(len(data)).mul(100).round(2),
})
print(rating_table.to_string())

expected_ratings = {
    "TV-Y", "TV-Y7", "TV-Y7-FV", "TV-G", "G", "PG", "TV-PG", "PG-13",
    "TV-14", "R", "TV-MA", "NC-17", "NR", "UR",
}
unexpected = data["rating"].notna() & ~data["rating"].isin(expected_ratings)
duration_like = data["rating"].str.fullmatch(r"\d+\s+min", na=False)
print(f"\nMissing ratings: {data['rating'].isna().sum()}")
print(f"Ratings outside the expected set: {unexpected.sum()}")
print(f"Duration-like values in rating: {duration_like.sum()}")
print("\nRows with unexpected ratings:")
print(df.loc[unexpected, ["title", "rating", "duration"]].to_string(index=False))


           Title count  All titles (%)
rating                                
TV-MA             3207           36.41
TV-14             2160           24.53
TV-PG              863            9.80
R                  799            9.07
PG-13              490            5.56
TV-Y7              334            3.79
TV-Y               307            3.49
PG                 287            3.26
TV-G               220            2.50
NR                  80            0.91
G                   41            0.47
TV-Y7-FV             6            0.07
<MISSING>            4            0.05
NC-17                3            0.03
UR                   3            0.03
74 min               1            0.01
84 min               1            0.01
66 min               1            0.01

Missing ratings: 4
Ratings outside the expected set: 3
Duration-like values in rating: 3

Rows with unexpected ratings:
                               title rating duration
                     Louis C.K. 2017 74 min   

## 5. Findings and preparation for modelling

- The CSV contains **8,807 titles**. Genre and description have no missing values in this version; **4 ratings are missing**.
- There are **42 individual genres** and **514 recorded genre combinations**. The most frequent individual genre is **International Movies**, occurring in **2,752 titles**.
- Descriptions have a mean length of **23.88 words** and a median of **24 words**. Their lengths range from **10 to 48 words**.
- The most common recorded rating is **TV-MA**, with **3,207 titles (36.41% of all rows)**. Unequal category sizes should be considered when evaluating a classifier.
- **3 rows contain duration-like values in `rating`**. Inspect these against the source and decide how to handle them, along with missing ratings and unrated labels, before training a model.

For later modelling, split individual genre labels for multi-label encoding, convert description text into numerical features such as word counts or TF-IDF, and define a consistent target-label policy. If ratings are grouped into broader categories, document that mapping as a project decision. Fit learned text/genre transformations on the training split to avoid information leakage. Compare predictions with a majority-class baseline and inspect per-class precision, recall, F1, and a confusion matrix.
